# Le prix de l'argent : le rendement américain à 10 ans · *The price of money: the U.S. 10-year yield*

Notebook compagnon du chapitre **8. La carte du voyage : de la COVID au soft landing (2020-2025)** — [lire l'article](https://nmlab.io/ressources/de-la-covid-au-soft-landing-2020-2025).
Companion notebook to chapter **8. The Map of the Journey: From COVID to the Soft Landing (2020-2025)** — [read the article](https://nmlab.io/en/ressources/from-covid-to-the-soft-landing-2020-2025).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_yield() -> Series:
    """Rendement quotidien du Treasury américain à 10 ans (DGS10), 2019-2025,
    en direct de FRED / Daily U.S. 10-year Treasury yield, live from FRED."""
    return nm.load_fred("DGS10", start="2019").loc["2019":"2025"].dropna()

dgs = load_yield()


from pandas import Timestamp as T
import matplotlib.dates as mdates
from matplotlib.figure import Figure
from matplotlib.ticker import FuncFormatter

LABELS = {
    "fr": dict(
        title="Le prix de l'argent : le rendement américain à 10 ans",
        peak="oct. 2023 : 5 % frôlés,\nplus haut depuis 2007",
        pivot="déc. 2023 : la Fed pivote,\nle 10 ans reflue",
        end="fin 2025 : 4,2 %",
        floor="4 août 2020 : 0,52 %",
        note="Source : Réserve fédérale via FRED (DGS10), rendement du Treasury à 10 ans, quotidien, 2019-2025.\n"
             "Plancher : 0,52 % le 4 août 2020 ; sommet : 4,98 % en clôture le 19 oct. 2023 — 5 % franchis en séance."),
    "en": dict(
        title="The price of money: the U.S. 10-year yield",
        peak="Oct. 2023: 5% brushed,\nhighest since 2007",
        pivot="Dec. 2023: the Fed pivots,\nthe 10-year eases",
        end="end 2025: 4.2%",
        floor="Aug. 4, 2020: 0.52%",
        note="Source: Federal Reserve via FRED (DGS10), 10-year Treasury yield, daily, 2019-2025.\n"
             "Floor: 0.52% on Aug. 4, 2020; peak: 4.98% at close on Oct. 19, 2023 — 5% crossed intraday."),
}

def build_figure(dgs: Series, lang: str) -> Figure:
    """Rendement à 10 ans, plancher de 2020, sommet et pivot de 2023 annotés."""
    text = LABELS[lang]
    pct = FuncFormatter(lambda v, _: (f"{v:.0f} %" if lang == "fr" else f"{v:.0f}%"))
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig, left=0.06, top=0.86)
    ax.plot(dgs.index, dgs, color=nm.COLORS["blue"], linewidth=1.9)

    ax.set_ylim(0, 6.3)
    ax.set_yticks(range(0, 7))
    ax.yaxis.set_major_formatter(pct)
    ax.set_xlim(T("2019-01-01"), T("2026-01-01"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    peak = dgs.loc["2023"].idxmax()
    floor = dgs.loc["2020"].idxmin()
    ax.annotate(text["peak"], xy=(peak, dgs.loc[peak]), xytext=(T("2021-06-01"), 5.25),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["pivot"], xy=(T("2023-12-27"), dgs.loc["2023-12-20":"2024-01-05"].mean()),
                xytext=(T("2024-01-01"), 2.55), fontsize=21, color=nm.COLORS["text"],
                va="center", ha="left", linespacing=1.5,
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.annotate(text["end"], xy=(dgs.index[-1], dgs.iloc[-1]), xytext=(T("2024-07-01"), 3.05),
                fontsize=21, color=nm.COLORS["text"], va="center", ha="left",
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.4))
    ax.text(T("2019-08-01"), 0.72, text["floor"], fontsize=21, color=nm.COLORS["text"], va="bottom", ha="left")
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(dgs, LANG)